# cluefin-xbrl 재무제표 분석 — 삼성전자 · 현대차

DART 에서 XBRL 사업보고서를 내려받아 `cluefin-xbrl` 로 파싱하는 예제입니다.
삼성전자(005930)와 제조기업인 현대차(005380) 두 종목을 분석합니다.

다루는 내용: 공시 검색으로 접수번호(rcept_no) 찾기 → XBRL 다운로드·파싱 →
연결/별도 본표(손익계산서·재무상태표) → 재무제표 주석(`extract_notes`) → 두 회사 비교.

## 사전 준비

- 리포지토리 루트의 `.env.test` 에 `DART_AUTH_KEY` 가 있어야 합니다 (공공 데이터라 계좌 위험 없음).
- 실행: 리포지토리 루트에서 `uv run --with jupyter jupyter lab` 후 이 노트북을 엽니다.
- XBRL ZIP 다운로드·파싱에 종목당 수십 초가 걸릴 수 있습니다.

## 1. DART 클라이언트와 corp_code 조회

In [ ]:
import os

import dotenv

from cluefin_openapi.dart._client import Client as DartClient
from cluefin_openapi.dart._periodic_report_financial_statement import PeriodicReportFinancialStatement
from cluefin_openapi.dart._public_disclosure import PublicDisclosure

dotenv.load_dotenv(dotenv.find_dotenv(".env.test", usecwd=True))

dart = DartClient(auth_key=os.environ["DART_AUTH_KEY"])
public_disclosure = PublicDisclosure(dart)
financial_statement = PeriodicReportFinancialStatement(dart)

TARGETS = {"삼성전자": "005930", "현대차": "005380"}
YEAR = "2025"  # 직전 사업연도 사업보고서
REPRT_CODE = "11011"  # 사업보고서

# corp_code 색인 (전체 기업 목록 zip — 최초 1회만 무겁다)
corp_index = {
    item.stock_code: item.corp_code
    for item in public_disclosure.corp_code().result.list
    if item.stock_code
}
corp_codes = {name: corp_index[code] for name, code in TARGETS.items()}
corp_codes

## 2. 사업보고서 접수번호(rcept_no) 찾기

보고서명에 유형 키워드(`사업보고서`)와 기간 마커(`(YYYY.12)`)가 모두 있어야
1분기(03)·3분기(09) 분기보고서와 구분됩니다.

In [ ]:
def find_rcept_no(corp_code: str, year: str) -> str | None:
    """연도별 사업보고서의 접수번호를 공시검색으로 찾는다."""
    result = public_disclosure.public_disclosure_search(
        corp_code=corp_code,
        bgn_de=f"{year}0101",
        end_de=f"{int(year) + 1}1231",  # 사업보고서는 이듬해 3월경 제출된다
        pblntf_ty="A",  # 정기공시
        last_reprt_at="Y",  # 정정공시가 있으면 최종본만
    )
    if result.result.status != "000":
        return None
    for item in result.result.list or []:
        if "사업보고서" in item.report_nm and f"({year}.12)" in item.report_nm:
            return item.rcept_no
    return None


rcept_nos = {name: find_rcept_no(cc, YEAR) for name, cc in corp_codes.items()}
rcept_nos

## 3. XBRL 다운로드 · 파싱

In [ ]:
import tempfile
from pathlib import Path

from cluefin_xbrl import extract_financial_statements, extract_notes, parse_xbrl_directory


def load_xbrl(rcept_no: str):
    dest = Path(tempfile.mkdtemp(prefix="cluefin_xbrl_"))
    xbrl_dir = financial_statement.download_financial_statement_xbrl(
        rcept_no=rcept_no,
        reprt_code=REPRT_CODE,
        destination=dest,
        overwrite=True,
    )
    doc = parse_xbrl_directory(xbrl_dir, include_taxonomy=True)
    return doc, extract_financial_statements(doc), extract_notes(doc)


docs = {}
for name, rcept_no in rcept_nos.items():
    doc, statements, notes = load_xbrl(rcept_no)
    docs[name] = {"doc": doc, "statements": statements, "notes": notes}
    print(
        f"{name}: facts {len(doc.facts):,} | 연결 본표 {list(statements.statements)} "
        f"| 별도 본표 {list(statements.separate_statements)} | 주석 {len(notes.notes)}건"
    )

## 4. 연결 손익계산서 — 두 회사 나란히 보기

In [ ]:
import pandas as pd

from cluefin_xbrl import statement_to_dicts


def statement_df(name: str, stmt_key: str, consolidated: bool = True) -> pd.DataFrame:
    stmts = docs[name]["statements"]
    source = stmts.statements if consolidated else stmts.separate_statements
    stmt = source.get(stmt_key)
    if stmt is None:
        return pd.DataFrame()
    df = pd.DataFrame(statement_to_dicts(stmt))
    return df


# IS(손익계산서)가 없는 회사는 CIS(포괄손익계산서)에 담겨 있을 수 있다
for name in docs:
    for key in ("IS", "CIS"):
        df = statement_df(name, key)
        if not df.empty:
            print(f"=== {name} — {key} (연결) 상위 12행 ===")
            display(df[[c for c in ("label_ko", "concept_local_name", "value", "unit", "period") if c in df.columns]].head(12))
            break

## 5. 연결 vs 별도 재무상태표 — 자산총계 비교

In [ ]:
def find_line(df: pd.DataFrame, keyword: str):
    if df.empty or "label_ko" not in df.columns:
        return None
    hit = df[df["label_ko"].fillna("").str.contains(keyword)]
    return hit.iloc[0]["value"] if not hit.empty else None


rows = []
for name in docs:
    rows.append(
        {
            "회사": name,
            "자산총계(연결)": find_line(statement_df(name, "BS", consolidated=True), "자산총계"),
            "자산총계(별도)": find_line(statement_df(name, "BS", consolidated=False), "자산총계"),
        }
    )
pd.DataFrame(rows)

## 6. 재무제표 주석 (`extract_notes`)

주석은 role 코드별 섹션으로 내려옵니다. 목차를 훑고, 관심 섹션 하나를 골라 항목을 봅니다.

In [ ]:
name = "삼성전자"
notes = docs[name]["notes"]

index = pd.DataFrame(
    {
        "role": n.role_code,
        "title": n.title,
        "연결여부": "연결" if n.is_consolidated else "별도",
        "items": len(n.line_items),
    }
    for n in sorted(notes.notes.values(), key=lambda n: n.role_code)
)
print(f"{name} 주석 {len(index)}건")
index.head(20)

In [ ]:
# 항목이 가장 많은 연결 주석 섹션 하나를 골라 값을 가진 항목 상위 15개를 본다
candidates = [n for n in notes.notes.values() if n.is_consolidated and n.line_items]
note = max(candidates, key=lambda n: len(n.line_items))
print(f"[{note.role_code}] {note.title} — {len(note.line_items)} items\n")

shown = 0
for item in note.line_items:
    if item.value is None or shown >= 15:
        continue
    label = item.label_ko or item.concept_local_name
    print(f"  {'  ' * item.depth}{label}: {item.value:,.0f} {item.unit or ''}")
    shown += 1